# Cypher 101 — Querying the Movie Graph

A walkthrough of the Cypher clauses you'll use in every query: `MATCH`, `WHERE`, `RETURN`, `WITH`, and multi-hop pattern traversal. Each section builds on the previous one and reuses a small recurring cast of anchors so you build intuition across examples.

### Prerequisites

This notebook **queries** a loaded graph but doesn't load anything itself. Run notebooks 03 → 07 first to set up the schema and load the data. The schema reference lives in [`docs/03_schema_diagram.md`](../docs/03_schema_diagram.md).

### Read-only notebook

Every cell below is a read query. For writes (`CREATE`, `MERGE`, `DELETE`) and how data gets *into* the graph, see notebooks 04 → 07.

### Recurring anchors

| Anchor | Used for |
|---|---|
| Toy Story (`movieId = 1`) | A universally recognizable Movie node |
| Tom Hanks | A well-connected Actor node |
| User 196 | A canonical MovieLens test user |
| Comedy | A high-coverage Genre |

### Cypher style conventions used throughout

- **Uppercase keywords** — `MATCH`, `WHERE`, `RETURN`, `WITH`, `LIMIT`
- **PascalCase labels** — `:Movie`, `:User`, `:Actor`
- **SCREAMING_SNAKE relationship types** — `:RATED`, `:DIRECTED_BY`, `:IN_GENRE`
- **lowercase variables** — `m`, `u`, `g`, `r`

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from graph import run_query, close_driver

## 1. MATCH — finding patterns in the graph

`MATCH` tells Cypher *what shape* of subgraph you want. The simplest pattern is a single node with a label and (optionally) an inline property filter.

In [2]:
result = run_query("""
MATCH (m:Movie {movieId: 1})
RETURN m.title, m.year
""")
for row in result:
    print(row)

{'m.title': 'Toy Story', 'm.year': 1995.0}


The real power of MATCH shows up when the pattern includes a relationship. The arrow direction matters — `(m:Movie)-[:IN_GENRE]->(g:Genre)` traverses *from* Movie *to* Genre, matching the direction the edge was loaded with.

You don't have to bind a variable to the relationship unless you want to read its properties. `[:IN_GENRE]` is enough when you only care that the edge exists.

In [3]:
result = run_query("""
MATCH (m:Movie {movieId: 1})-[:IN_GENRE]->(g:Genre)
RETURN g.name
""")
for row in result:
    print(row["g.name"])

Animation
Children's
Comedy


## 2. WHERE — filtering beyond inline property match

There are two ways to filter. The inline form `(m:Movie {year: 1995})` is handy for **equality** on a single property. For everything else — ranges, comparisons, multiple conditions, regex — use `WHERE`.

A useful rule of thumb: anything you *could* express inline can also be a `WHERE` clause. The reverse isn't true.

In [4]:
result = run_query("""
MATCH (m:Movie)-[:IN_GENRE]->(:Genre {name: "Comedy"})
WHERE m.year = 1995
RETURN m.title
LIMIT 5
""")
for row in result:
    print(row["m.title"])

Toy Story
Get Shorty
Babe
Mighty Aphrodite
French Twist (Gazon maudit)


In [5]:
result = run_query("""
MATCH (u:User {userId: 196})-[r:RATED]->(m:Movie)
WHERE r.rating >= 4
RETURN m.title, r.rating
LIMIT 10
""")
for row in result:
    print(f"{row['m.title']:<35} {row['r.rating']}")

Babe                                5.0
Birdcage, The                       4.0
Ace Ventura: Pet Detective          5.0
Kids in the Hall: Brain Candy       4.0
Truth About Cats & Dogs, The        4.0
Fish Called Wanda, A                5.0
Raising Arizona                     4.0
Secrets & Lies                      5.0
English Patient, The                5.0
Mrs. Brown (Her Majesty, Mrs. Brown) 4.0


## 3. RETURN — projection, aliases, ordering

`RETURN` is the final shape of your result. Three things commonly happen here:

- **Projection** — pick which properties you want (`m.title`, `m.year`)
- **Aliases** with `AS` — rename columns for readability (`m.title AS movie`)
- **`ORDER BY` / `LIMIT`** — control row order and how many come back

Aliasing matters more than it looks. Without it, downstream Python code has to read awkward keys like `row["m.title"]`. With `AS movie`, you get `row["movie"]`.

In [6]:
result = run_query("""
MATCH (a:Actor {name: "Tom Hanks"})<-[:FEATURES_ACTOR]-(m:Movie)
RETURN m.title AS movie, m.year AS released
ORDER BY released DESC
LIMIT 10
""")
for row in result:
    print(f"{row['released']}  {row['movie']}")

1996.0  That Thing You Do!
1995.0  Toy Story
1995.0  Apollo 13
1994.0  Forrest Gump
1993.0  Sleepless in Seattle
1993.0  Philadelphia


## 4. WITH — the pipeline operator

`WITH` is the pivot between two stages of a query. Anything before `WITH` is *one stage*; anything after is a *new stage* that only sees what `WITH` forwarded.

This unlocks something `MATCH + WHERE` alone can't do: **filter on an aggregate**. In SQL this would be a subquery or `HAVING`. In Cypher it's just `WITH ... aggregate-here ... WHERE ...`.

Below: find power-raters — users who rated more than 200 movies.

In [7]:
result = run_query("""
MATCH (u:User)-[:RATED]->(m:Movie)
WITH u, count(m) AS num_ratings
WHERE num_ratings > 200
RETURN u.userId AS user, num_ratings
ORDER BY num_ratings DESC
LIMIT 10
""")
for row in result:
    print(f"User {row['user']:<5} {row['num_ratings']} ratings")

User 405   737 ratings
User 655   685 ratings
User 13    636 ratings
User 450   540 ratings
User 276   518 ratings
User 416   493 ratings
User 537   490 ratings
User 303   484 ratings
User 234   480 ratings
User 393   448 ratings


### `collect()` — the list-aggregation half of `WITH`

`count()` collapses a group of rows into a single number. `collect()` collapses them into a **list**. Both are aggregations — Cypher infers the grouping key automatically from the non-aggregated variables in `WITH`.

`collect()` is load-bearing for explainability paths later — a recommendation like *"you rated Inception 5★ → Nolan → The Prestige"* is built from `collect`-ed paths.

Below: for each user with more than 20 five-star ratings, collect the movie titles into a list and show the first three.

In [8]:
result = run_query("""
MATCH (u:User)-[r:RATED {rating: 5}]->(m:Movie)
WITH u, collect(m.title) AS five_star_titles, count(m) AS total
WHERE total > 20
RETURN u.userId AS user, five_star_titles[0..3] AS sample, total
ORDER BY total DESC
LIMIT 5
""")
for row in result:
    print(f"User {row['user']:<5} ({row['total']} fives)  sample: {row['sample']}")

User 416   (172 fives)  sample: ['Toy Story', 'Star Wars', 'Pulp Fiction']
User 7     (161 fives)  sample: ['Star Wars', 'Pulp Fiction', 'Fargo']
User 90    (147 fives)  sample: ['Pulp Fiction', 'Fargo', 'Babe']
User 592   (145 fives)  sample: ['Star Wars', 'Pulp Fiction', 'Fargo']
User 747   (142 fives)  sample: ['Toy Story', 'Star Wars', 'Pulp Fiction']


## 5. Multi-hop traversals

This is where the graph model earns its keep. A two-hop query — `MATCH (a)-[:R1]->(b)-[:R2]->(c)` — is one line in Cypher and three self-joins in SQL.

Three shapes worth knowing:

1. **Collaborative shape** — two users connected via a shared movie
2. **Content shape** — two movies connected via a shared actor
3. **3-hop shape** — a user, a movie they rated, a genre, and other movies in that genre

We start with shape 1 — and there's a subtle Cypher behavior to spot first.

### Shape 1 — Collaborative (2-hop)

> *Find users who rated the same movies as user 196.*

The query below has two `:User` slots, anchored at user 196 on one side and unconstrained on the other. At a glance it looks like it could match user 196 to themselves — every movie user 196 rated has them as the rater on both ends of the pattern. Run it. Does it?

In [9]:
result = run_query("""
MATCH (u1:User {userId: 196})-[:RATED]->(m:Movie)<-[:RATED]-(u2:User)               
RETURN u1.userId AS user_1, u2.userId AS user_2, count(m) AS shared
ORDER BY shared DESC
LIMIT 5
""")
for row in result:
    print(f"user_1={row['user_1']:<4} user_2={row['user_2']:<4} shared={row['shared']}")

user_1=196  user_2=655  shared=31
user_1=196  user_2=13   shared=30
user_1=196  user_2=378  shared=28
user_1=196  user_2=303  shared=28
user_1=196  user_2=416  shared=26


**Where's user 196 in the results?**

If the self-match were happening, the top row would have been `user_1=196, user_2=196` with `shared=31` (every movie user 196 rated). It isn't there at all. Cypher silently prevented it — and the reason is a feature worth knowing.

**Relationship uniqueness.** Within a single `MATCH` pattern, every relationship slot must bind to a *distinct* edge. Our pattern has two `:RATED` slots:

`(u1)-[:RATED]->(m)<-[:RATED]-(u2)`

If `u2` bound to user 196, both `:RATED` slots would have to point to the same physical edge — the single `:RATED` edge between user 196 and movie `m` (our schema has one per user-movie pair). Cypher rejects that binding outright.

This protection kicks in any time the path between two same-label nodes is short enough that the relationship slots would collapse onto a single edge.

**It isn't a guarantee, though.** In a longer path — e.g. `(u1)-[:RATED]->(m1)-[:IN_GENRE]->(g)<-[:IN_GENRE]-(m2)<-[:RATED]-(u2)` — the two `:RATED` slots go to *different* movies, so they're distinct edges by construction. `u1 = u2` becomes a valid binding again, and you do need `WHERE u1 <> u2` to filter it out.

The next cell adds `WHERE u1 <> u2` and projects only the neighbor. In this pattern the filter is redundant, but writing it explicitly is a good habit — it signals intent and protects you when the pattern grows.

In [10]:
result = run_query("""
MATCH (u1:User {userId: 196})-[:RATED]->(m:Movie)<-[:RATED]-(u2:User)
WHERE u1 <> u2
RETURN u2.userId AS neighbor, count(m) AS shared
ORDER BY shared DESC
LIMIT 5
""")
for row in result:
    print(f"User {row['neighbor']:<5} shared {row['shared']} movies with user 196")

User 655   shared 31 movies with user 196
User 13    shared 30 movies with user 196
User 378   shared 28 movies with user 196
User 303   shared 28 movies with user 196
User 416   shared 26 movies with user 196


### Shape 2 — Content (2-hop) via shared actor

> *Find other movies featuring any actor from Toy Story.*

Same two-hop structure, different semantic shape — instead of going through a User, we go through an Actor. Relationship uniqueness would block `other = m` here for the same reason it blocked `u1 = u2` in Shape 1 (both `:FEATURES_ACTOR` slots would collapse onto a single edge), so the `WHERE other <> m` filter below is defensive rather than load-bearing — it makes intent explicit without changing the result set.

Notice the result connects both recurring anchors in one query: Toy Story → Tom Hanks → his other films.

In [11]:
result = run_query("""
MATCH (m:Movie {movieId: 1})-[:FEATURES_ACTOR]->(a:Actor)<-[:FEATURES_ACTOR]-(other:Movie)
WHERE other <> m
RETURN a.name AS actor, other.title AS other_film, other.year AS year
ORDER BY year
LIMIT 10
""")
for row in result:
    print(f"{row['year']}  {row['other_film']:<40} ({row['actor']})")

1987.0  Princess Bride, The                      (Wallace Shawn)
1993.0  Sleepless in Seattle                     (Tom Hanks)
1993.0  Philadelphia                             (Tom Hanks)
1993.0  Beverly Hillbillies, The                 (Jim Varney)
1994.0  Santa Clause, The                        (Tim Allen)
1994.0  Mrs. Parker and the Vicious Circle       (Wallace Shawn)
1994.0  Forrest Gump                             (Tom Hanks)
1994.0  Vanya on 42nd Street                     (Wallace Shawn)
1994.0  Blink                                    (Laurie Metcalf)
1995.0  Apollo 13                                (Tom Hanks)


### Shape 3 — 3-hop genre traversal

> *Find movies in the genres user 196 has already rated.*

Now we chain three hops: User → Movie → Genre → Movie. One subtlety: the same recommended movie can match through *multiple* shared genres — a film tagged Comedy *and* Drama that lines up with two of user 196's rated movies will appear twice in the result.

`DISTINCT` collapses the duplicates. It's a per-row deduplication on the `RETURN` clause, not an aggregation. Without it you'd have to deduplicate in Python.

(For an extended version of this shape — three actors who have *all* worked with each other, the triangle pattern — see notebook 07.)

In [12]:
result = run_query("""
MATCH (u:User {userId: 196})-[:RATED]->(m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
WHERE rec <> m
RETURN DISTINCT rec.title AS recommendation
LIMIT 10
""")
for row in result:
    print(row["recommendation"])

Toy Story
Free Willy 2: The Adventure Home
Santa Clause, The
Lion King, The
Free Willy
Nightmare Before Christmas, The
Home Alone
Aladdin
Snow White and the Seven Dwarfs
Aristocats, The


## 6. Putting it together — top 5 most-rated Comedies

One query that uses every clause from §1–§4: `MATCH`, `WHERE`, `WITH` with two aggregations (`count` and `avg`), filter on the aggregate, then `RETURN` with `ORDER BY` and `LIMIT`.

This is a **popularity** query, not a personalized recommendation — the recommender strategies (T-19 onward) are what get personalized. The point here is the shape of a real analytical query that touches everything you've seen so far.

In [13]:
result = run_query("""
MATCH (m:Movie)-[:IN_GENRE]->(:Genre {name: "Comedy"})
MATCH (m)<-[r:RATED]-()
WITH m, count(r) AS num_ratings, avg(r.rating) AS avg_rating
WHERE num_ratings > 50
RETURN m.title AS movie, num_ratings, round(avg_rating, 2) AS avg
ORDER BY num_ratings DESC
LIMIT 5
""")
for row in result:
    print(f"{row['movie']:<35} ratings={row['num_ratings']:<5} avg={row['avg']}")

Liar Liar                           ratings=485   avg=3.16
Toy Story                           ratings=452   avg=3.88
Back to the Future                  ratings=350   avg=3.83
Willy Wonka and the Chocolate Factory ratings=326   avg=3.63
Princess Bride, The                 ratings=324   avg=4.17


In [14]:
close_driver()